# 06 · Final run and report

This notebook produces the published results. It regenerates every table and figure from a single version
of the code, writes the headline findings and figure into the README, and builds the results page of the
project website. Every number in the README findings and on the results page comes from this run.

In [ ]:
# Make the project importable; works in JupyterLab, VS Code and Colab without reinstalling.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from mapaudit import notebook as nb
from mapaudit import pipeline, report
from mapaudit.config import get_paths

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:,.3f}".format)
MODE = nb.notebook_mode()  # "real" by default; CI sets MAPAUDIT_NOTEBOOK_MODE=demo (synthetic data)
print("Mode:", MODE)

## 1. Regenerate everything (final clean run, about 20 minutes)

In [ ]:
RERUN_ALL = False  # set to True for the final run
if MODE == "demo":
    nb.prepare_demo(through="simulate")
if RERUN_ALL:
    for step in (pipeline.build, pipeline.audit, pipeline.robustness, pipeline.simulate):
        step(MODE)
report.report(MODE)

## 2. Headline findings

The headline sentences are generated from the result tables, so every number in them comes from this run.
The cell also writes them, with the headline figure, into the findings section of the README.

In [ ]:
print((get_paths(MODE).results / "headline.md").read_text())
if MODE == "real":
    nb.update_readme()
    print("README.md updated with the headline findings and figure.")

## 3. Final checks

The full check suite (formatting, lint, tests and the documentation build), plus a release check that the
real-data results are present and no draft placeholders remain in the documentation.

In [ ]:
if MODE == "real":
    display(nb.run_checks(fix=True, release=True))